# Error-reparameterised Predictive Coding on CIFAR-10 with an MLP-Mixer

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thebuckleylab/jpc/blob/main/examples/epc_cifar_mixer.ipynb)

This notebook is a variant of the [ePC](epc.ipynb) example ([Goemaere et al., 2025](https://openreview.net/forum?id=lQhBWz59qW)). Instead of a standard MLP on MNIST, we train an error-reparameterised PC network with an **MLP-Mixer** ([Tolstikhin et al., 2021](https://arxiv.org/abs/2105.01601)) on **CIFAR-10**.

The Mixer is built as a list of callables that `jpc` can treat as predictive-coding layers: a convolutional **patch-embedding stem** that maps the image to a `(num_patches, hidden_dim)` token table, a stack of **Mixer blocks** (token-mixing MLP + channel-mixing MLP, each with LayerNorm and a skip connection), and a final **classifier head** (LayerNorm + global average pooling + linear) that maps the tokens to class logits. The ePC update functions do not care that layer activities are 2D per sample `(num_patches, hidden_dim)` rather than 1D — the errors are just initialised with the matching shape.

In [ ]:
%%capture
!pip install torch==2.3.1
!pip install torchvision==0.18.1

In [9]:
import jpc

import jax
import jax.numpy as jnp
import jax.random as jr
import equinox as eqx
import equinox.nn as nn
import optax

import numpy as np

import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import warnings
warnings.simplefilter('ignore')  # ignore warnings

## Hyperparameters

The Mixer has **6 layers** for ePC purposes: 1 patch-embedding stem + `N_BLOCKS` Mixer blocks + 1 classifier head. With `PATCH_SIZE = 4` and `IMG_SIZE = 32` the token table has `S = (32 / 4)**2 = 64` patches of `HIDDEN_DIM = 128` channels each.

| Activity | Shape |
| --- | --- |
| `activities[0]` (= patch embeddings) | `(64, 128)` |
| `activities[1..N_BLOCKS]` | `(64, 128)` |
| `activities[-1]` (= class logits) | `(10,)` |

Images are normalised with the standard CIFAR mean/std — the Mixer head outputs raw logits so the input distribution doesn't need to be clipped.

In [10]:
SEED = 0

N_CLASSES = 10
IMG_CHANNELS = 3
IMG_SIZE = 32

PATCH_SIZE = 4
HIDDEN_DIM = 128
TOKEN_MLP_DIM = 64
CHANNEL_MLP_DIM = 256
N_BLOCKS = 4
ACT_FN = "gelu"
USE_BIAS = True

STATE_LR = 1e-1       # error optimiser learning rate
PARAM_LR = 1e-3
BATCH_SIZE = 64
TEST_EVERY = 50
N_TRAIN_ITERS = 3000

# inference iterations per batch
N_INFERENCE_STEPS = 8

## Dataset

Some utils to fetch CIFAR-10. Unlike the MNIST ePC example, images are **not** flattened — they are kept as `(3, 32, 32)` tensors so that the convolutional patch-embedding stem can operate directly on them.

In [11]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)
CIFAR_CLASSES = (
    'plane', 'car', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
)


def get_cifar_loaders(batch_size):
    train_data = CIFAR10(train=True, normalise=True)
    test_data = CIFAR10(train=False, normalise=True)
    train_loader = DataLoader(
        dataset=train_data,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True
    )
    test_loader = DataLoader(
        dataset=test_data,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True
    )
    return train_loader, test_loader


class CIFAR10(datasets.CIFAR10):
    def __init__(self, train, normalise=True, save_dir="data"):
        if normalise:
            transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD)
            ])
        else:
            transform = transforms.Compose([transforms.ToTensor()])
        super().__init__(save_dir, download=True, train=train, transform=transform)

    def __getitem__(self, index):
        img, label = super().__getitem__(index)  # img has shape (3, 32, 32)
        label = one_hot(label)
        return img, label


def one_hot(labels, n_classes=10):
    arr = torch.eye(n_classes)
    return arr[labels]

## MLP-Mixer model

We build the Mixer as a plain Python list of `equinox` modules, matching what [`jpc.make_mlp()`](https://thebuckleylab.github.io/jpc/api/Utils/#jpc.make_mlp) returns — the ePC update functions don't care whether each layer is fully-connected, convolutional, or a Mixer block. Each module is written to take a **single sample** as input; `jpc` internally `vmap`s each layer over the batch dimension.

A Mixer block is two residual sub-blocks:

1. **Token-mixing MLP**: `LayerNorm → transpose → MLP(S → D_S → S) → transpose → + skip`. The MLP sees one channel's values across all tokens at a time.
2. **Channel-mixing MLP**: `LayerNorm → MLP(C → D_C → C) → + skip`. The MLP sees one token's channel vector at a time.

Both MLPs use GELU and have 2 linear layers. The token-mixing MLP is `vmap`ped across channels, and the channel-mixing MLP is `vmap`ped across tokens — this is what realises the paper's parameter sharing.

In [12]:
def get_act(name):
    return jpc.get_act_fn(name)


class MlpBlock(eqx.Module):
    linear1: nn.Linear
    linear2: nn.Linear
    act_name: str = eqx.field(static=True)

    def __init__(self, in_dim, hidden_dim, out_dim, use_bias, act_name, key):
        k1, k2 = jr.split(key, 2)
        self.linear1 = nn.Linear(in_dim, hidden_dim, use_bias=use_bias, key=k1)
        self.linear2 = nn.Linear(hidden_dim, out_dim, use_bias=use_bias, key=k2)
        self.act_name = act_name

    def __call__(self, x, *, key=None):
        y = self.linear1(x)
        y = get_act(self.act_name)(y)
        return self.linear2(y)


class MixerBlock(eqx.Module):
    ln1: nn.LayerNorm
    ln2: nn.LayerNorm
    token_mlp: MlpBlock
    channel_mlp: MlpBlock

    def __init__(
        self,
        num_patches,
        hidden_dim,
        tokens_mlp_dim,
        channels_mlp_dim,
        use_bias,
        act_name,
        key
    ):
        k_tok, k_chan = jr.split(key, 2)
        self.ln1 = nn.LayerNorm(shape=(hidden_dim,))
        self.ln2 = nn.LayerNorm(shape=(hidden_dim,))
        self.token_mlp = MlpBlock(
            in_dim=num_patches,
            hidden_dim=tokens_mlp_dim,
            out_dim=num_patches,
            use_bias=use_bias,
            act_name=act_name,
            key=k_tok
        )
        self.channel_mlp = MlpBlock(
            in_dim=hidden_dim,
            hidden_dim=channels_mlp_dim,
            out_dim=hidden_dim,
            use_bias=use_bias,
            act_name=act_name,
            key=k_chan
        )

    def __call__(self, x, *, key=None):
        # x: (S, C) for a single sample.
        # token-mixing: LayerNorm per token, then one MLP(S -> S) per channel.
        y = jax.vmap(self.ln1)(x)            # (S, C)
        y = jnp.swapaxes(y, 0, 1)             # (C, S)
        y = jax.vmap(self.token_mlp)(y)       # (C, S)
        y = jnp.swapaxes(y, 0, 1)             # (S, C)
        x = x + y
        # channel-mixing: LayerNorm per token, then one MLP(C -> C) per token.
        y = jax.vmap(self.ln2)(x)             # (S, C)
        y = jax.vmap(self.channel_mlp)(y)     # (S, C)
        return x + y


class PatchEmbed(eqx.Module):
    conv: nn.Conv2d

    def __init__(self, img_channels, hidden_dim, patch_size, use_bias, key):
        self.conv = nn.Conv2d(
            in_channels=img_channels,
            out_channels=hidden_dim,
            kernel_size=patch_size,
            stride=patch_size,
            use_bias=use_bias,
            key=key
        )

    def __call__(self, x, *, key=None):
        # x: (C_in, H, W) for a single sample.
        y = self.conv(x)                  # (C, H/P, W/P)
        C, H, W = y.shape
        # (C, H, W) -> (H, W, C) -> (H * W, C)
        return jnp.transpose(y, (1, 2, 0)).reshape(H * W, C)


class MixerHead(eqx.Module):
    ln: nn.LayerNorm
    classifier: nn.Linear

    def __init__(self, hidden_dim, n_classes, use_bias, key):
        self.ln = nn.LayerNorm(shape=(hidden_dim,))
        self.classifier = nn.Linear(
            hidden_dim, n_classes, use_bias=use_bias, key=key
        )

    def __call__(self, x, *, key=None):
        # x: (S, C) for a single sample.
        y = jax.vmap(self.ln)(x)       # (S, C), per-token layer norm
        y = jnp.mean(y, axis=0)         # (C,), global average pool over tokens
        return self.classifier(y)        # (n_classes,)


def make_mlp_mixer(
    key,
    img_channels,
    img_size,
    patch_size,
    hidden_dim,
    tokens_mlp_dim,
    channels_mlp_dim,
    n_blocks,
    n_classes,
    use_bias,
    act_name
):
    num_patches = (img_size // patch_size) ** 2
    keys = jr.split(key, n_blocks + 2)

    layers = [
        PatchEmbed(
            img_channels=img_channels,
            hidden_dim=hidden_dim,
            patch_size=patch_size,
            use_bias=use_bias,
            key=keys[0]
        )
    ]
    for i in range(n_blocks):
        layers.append(MixerBlock(
            num_patches=num_patches,
            hidden_dim=hidden_dim,
            tokens_mlp_dim=tokens_mlp_dim,
            channels_mlp_dim=channels_mlp_dim,
            use_bias=use_bias,
            act_name=act_name,
            key=keys[1 + i]
        ))
    layers.append(MixerHead(
        hidden_dim=hidden_dim,
        n_classes=n_classes,
        use_bias=use_bias,
        key=keys[-1]
    ))
    return layers

## Train and test

Since ePC layer activities here are 2D per sample `(num_patches, hidden_dim)` rather than 1D, we can't use [`jpc.init_epc_errors()`](https://thebuckleylab.github.io/jpc/api/Initialisation/#jpc.init_epc_errors) — that helper assumes flat per-sample layer sizes. We instead do a forward pass to get the correct shapes and initialise a list of zero-filled errors that matches the activities element-wise. The ePC energy just needs each `errors[l]` to be broadcastable against `model[l](current_activity)`, which this guarantees.

Inference and parameter updates are then identical to the MLP case: we call [`jpc.update_epc_errors()`](https://thebuckleylab.github.io/jpc/api/Discrete%20updates/#jpc.update_epc_errors) in a loop, then one call to [`jpc.update_epc_params()`](https://thebuckleylab.github.io/jpc/api/Discrete%20updates/#jpc.update_epc_params) per batch.

In [13]:
def init_epc_errors_like(activities):
    """Zero errors matching each layer activity's shape (incl. the batch dim)."""
    return [jnp.zeros_like(a) for a in activities]


def evaluate(model, test_loader):
    avg_test_acc = 0.
    for _, (img_batch, label_batch) in enumerate(test_loader):
        img_batch, label_batch = img_batch.numpy(), label_batch.numpy()
        _, test_acc = jpc.test_discriminative_pc(
            model=model,
            input=img_batch,
            output=label_batch
        )
        avg_test_acc += test_acc
    return avg_test_acc / len(test_loader)


def train(
    seed,
    img_channels,
    img_size,
    patch_size,
    hidden_dim,
    tokens_mlp_dim,
    channels_mlp_dim,
    n_blocks,
    n_classes,
    use_bias,
    act_name,
    state_lr,
    param_lr,
    batch_size,
    test_every,
    n_train_iters,
    n_inference_steps
):
    key = jr.PRNGKey(seed)
    model = make_mlp_mixer(
        key=key,
        img_channels=img_channels,
        img_size=img_size,
        patch_size=patch_size,
        hidden_dim=hidden_dim,
        tokens_mlp_dim=tokens_mlp_dim,
        channels_mlp_dim=channels_mlp_dim,
        n_blocks=n_blocks,
        n_classes=n_classes,
        use_bias=use_bias,
        act_name=act_name
    )

    error_optim = optax.sgd(state_lr)
    param_optim = optax.adam(param_lr)
    param_opt_state = param_optim.init(
        (eqx.filter(model, eqx.is_array), None)
    )

    train_loader, test_loader = get_cifar_loaders(batch_size)

    iter = 0
    epoch = 0
    done = False
    while not done:
        epoch += 1
        for img_batch, label_batch in train_loader:
            img_batch, label_batch = img_batch.numpy(), label_batch.numpy()

            # shape-matched zero initial errors, derived from a feed-forward pass
            activities = jpc.init_activities_with_ffwd(
                model=model,
                input=img_batch
            )
            train_loss = jpc.mse_loss(activities[-1], label_batch)

            errors = init_epc_errors_like(activities)
            error_opt_state = error_optim.init(errors)

            # inference: update errors to minimise the ePC energy
            for _ in range(n_inference_steps):
                error_update_result = jpc.update_epc_errors(
                    params=(model, None),
                    errors=errors,
                    optim=error_optim,
                    opt_state=error_opt_state,
                    output=label_batch,
                    input=img_batch
                )
                errors = error_update_result["errors"]
                error_opt_state = error_update_result["opt_state"]

            # learning: one parameter update given the inferred errors
            param_update_result = jpc.update_epc_params(
                params=(model, None),
                errors=errors,
                optim=param_optim,
                opt_state=param_opt_state,
                output=label_batch,
                input=img_batch
            )
            model = param_update_result["model"]
            param_opt_state = param_update_result["opt_state"]

            iter += 1

            if np.isinf(train_loss) or np.isnan(train_loss):
                print(
                    f"Stopping training because of divergence, train loss={train_loss}"
                )
                done = True
                break

            if (iter % test_every) == 0:
                avg_test_acc = evaluate(model=model, test_loader=test_loader)
                print(
                    f"Epoch {epoch}, train iter {iter}, "
                    f"train loss={train_loss:4f}, "
                    f"avg test accuracy={avg_test_acc:4f}"
                )

            if iter >= n_train_iters:
                done = True
                break

    return model

## Run

In [14]:
model = train(
    seed=SEED,
    img_channels=IMG_CHANNELS,
    img_size=IMG_SIZE,
    patch_size=PATCH_SIZE,
    hidden_dim=HIDDEN_DIM,
    tokens_mlp_dim=TOKEN_MLP_DIM,
    channels_mlp_dim=CHANNEL_MLP_DIM,
    n_blocks=N_BLOCKS,
    n_classes=N_CLASSES,
    use_bias=USE_BIAS,
    act_name=ACT_FN,
    state_lr=STATE_LR,
    param_lr=PARAM_LR,
    batch_size=BATCH_SIZE,
    test_every=TEST_EVERY,
    n_train_iters=N_TRAIN_ITERS,
    n_inference_steps=N_INFERENCE_STEPS
)

Epoch 1, train iter 50, train loss=0.414562, avg test accuracy=30.508814
Epoch 1, train iter 100, train loss=0.397574, avg test accuracy=37.730370
Epoch 1, train iter 150, train loss=0.352808, avg test accuracy=39.312901
Epoch 1, train iter 200, train loss=0.378039, avg test accuracy=40.775242
Epoch 1, train iter 250, train loss=0.372138, avg test accuracy=42.528046
Epoch 1, train iter 300, train loss=0.337984, avg test accuracy=44.300880
Epoch 1, train iter 350, train loss=0.351786, avg test accuracy=46.584534
Epoch 1, train iter 400, train loss=0.362745, avg test accuracy=46.794872
Epoch 1, train iter 450, train loss=0.339305, avg test accuracy=48.487579
Epoch 1, train iter 500, train loss=0.363971, avg test accuracy=48.016827
Epoch 1, train iter 550, train loss=0.300598, avg test accuracy=51.031651
Epoch 1, train iter 600, train loss=0.343981, avg test accuracy=51.582531
Epoch 1, train iter 650, train loss=0.349996, avg test accuracy=52.604168
Epoch 1, train iter 700, train loss=0.2

## Ablation: bigger Mixer

The four levers that scale Mixer capacity are:

- **`HIDDEN_DIM`** — per-token channel width `C`. Sets the size of every activity `(S, C)` and the width of the channel-mixing MLP. Dominant width knob.
- **`N_BLOCKS`** — depth. Each extra block adds a token-mix + channel-mix pair *and* one more activity per inference step (so ePC inference cost grows linearly).
- **`TOKEN_MLP_DIM`** — hidden dim of the token-mixing MLP (`S → D_S → S`, applied per channel).
- **`CHANNEL_MLP_DIM`** — hidden dim of the channel-mixing MLP (`C → D_C → C`, applied per token). The paper uses `D_C ≈ 4·C`.

For reference, [Tolstikhin et al., 2021](https://arxiv.org/abs/2105.01601) use `(N_BLOCKS, HIDDEN_DIM, TOKEN_MLP_DIM, CHANNEL_MLP_DIM) = (8, 512, 256, 2048)` for Mixer-S, scaling all four together for Mixer-B/L.

The ablation below scales the same way — twice as deep (`8` vs `4` blocks) and 50% wider (`192` vs `128` channels, `96` vs `64` for token-mixing, `512` vs `256` for channel-mixing). Same `N_TRAIN_ITERS` so the comparison is on per-iteration data (not wall-clock).

In [15]:
BIG_HIDDEN_DIM = 192
BIG_TOKEN_MLP_DIM = 96
BIG_CHANNEL_MLP_DIM = 512
BIG_N_BLOCKS = 8

big_model = train(
    seed=SEED,
    img_channels=IMG_CHANNELS,
    img_size=IMG_SIZE,
    patch_size=PATCH_SIZE,
    hidden_dim=BIG_HIDDEN_DIM,
    tokens_mlp_dim=BIG_TOKEN_MLP_DIM,
    channels_mlp_dim=BIG_CHANNEL_MLP_DIM,
    n_blocks=BIG_N_BLOCKS,
    n_classes=N_CLASSES,
    use_bias=USE_BIAS,
    act_name=ACT_FN,
    state_lr=STATE_LR,
    param_lr=PARAM_LR,
    batch_size=BATCH_SIZE,
    test_every=TEST_EVERY,
    n_train_iters=N_TRAIN_ITERS,
    n_inference_steps=N_INFERENCE_STEPS
)

Epoch 1, train iter 50, train loss=0.417517, avg test accuracy=22.005209
Epoch 1, train iter 100, train loss=0.408407, avg test accuracy=29.467148
Epoch 1, train iter 150, train loss=0.397971, avg test accuracy=34.124599
Epoch 1, train iter 200, train loss=0.370410, avg test accuracy=36.588543
Epoch 1, train iter 250, train loss=0.398517, avg test accuracy=39.423077
Epoch 1, train iter 300, train loss=0.333695, avg test accuracy=41.386219
Epoch 1, train iter 350, train loss=0.369273, avg test accuracy=44.871796
Epoch 1, train iter 400, train loss=0.371877, avg test accuracy=46.384216
Epoch 1, train iter 450, train loss=0.337706, avg test accuracy=46.915066
Epoch 1, train iter 500, train loss=0.372570, avg test accuracy=46.274040
Epoch 1, train iter 550, train loss=0.330336, avg test accuracy=48.918270
Epoch 1, train iter 600, train loss=0.354468, avg test accuracy=47.475960
Epoch 1, train iter 650, train loss=0.352276, avg test accuracy=50.971554
Epoch 1, train iter 700, train loss=0.3

CIFAR-10 is much harder than MNIST. The training loop now restarts the dataloader as needed and keeps going until `N_TRAIN_ITERS` iterations have been reached, so `N_TRAIN_ITERS` controls training length across as many epochs as required (one epoch at batch 64 is ~781 iterations over the 50 k training set). For a longer run, bump `N_TRAIN_ITERS` and increase `N_BLOCKS` / `HIDDEN_DIM` to get closer to the configurations in [Tolstikhin et al., 2021](https://arxiv.org/abs/2105.01601).